### Meeting on 

# Does concept frequency predict whether few-shot adaptation helps?

**Self-contained.** Everything from training to the final statistics is in this one file. No
number is pasted from another notebook or from chat — all of it is computed from data on disk.

## The claim, in four steps

| | | Status |
|---|---|---|
| 1 | Zero-shot VLMs are not truly zero-shot: the pathology *names* appear in the training reports | prior work, we measure it here |
| 2 | Pathologies named **rarely** score **worse** zero-shot | replication (Udandarao et al. 2024, in radiology) |
| 3 | So we add K labeled examples — few-shot adaptation | method |
| 4 | Adaptation helps the **rare** ones and **hurts** the common ones | **our contribution** |

Step 4 is the part that is new. Stated precisely, it is a sign flip:

$$\text{AUC}_{\text{zero-shot}} = +a\,\log_{10}(f) + b \qquad\qquad \Delta\text{AUC}_{\text{few-shot}} = -c\,\log_{10}(f) + d$$

where $f$ is how often the pathology is named in the training reports. **The two lines cross**,
and the crossing is a label-free rule for deciding where to adapt.

## Design decisions, and why

**Single checkpoint, no ensemble.** CheXzero ensembles its top 10. We use the single best. This
costs accuracy — their own figures are 0.878 single vs 0.889 ensembled — but removes a moving
part, so our reproduction target is **0.878**, not 0.889.

**Each dataset selects its own checkpoint.** An earlier run ranked checkpoints on CheXpert
validation and applied them to ChestX-ray14 as well, which handed CheXpert a home-field
advantage and made the cross-dataset numbers indefensible. Here:

```
CheXpert  →  select on CheXpert val (200 studies)   →  support from CheXpert train
NIH       →  select on held-out NIH split (6,000)   →  support from the rest of promptdev
```

The cost is that the two datasets now use **different checkpoints**, so within-dataset results
are primary and any cross-dataset comparison is secondary. That trade is the right way round:
the hypothesis is about variation *across pathologies*, which lives inside each dataset.

## Runtime

| Stage | Fresh | With cache |
|---|---|---|
| Train 8 epochs on MIMIC | ~1h35m | skipped |
| Checkpoint selection, both datasets | ~55m | skipped |
| Encode + all statistics | ~5m | ~1m |

Every expensive step writes a cache and is skipped on re-run. Safe to run top to bottom twice.

**Kernel:** the `hj` conda environment (`/home/hj/anaconda3/envs/hj/bin/python`).

In [ ]:
import os, re, sys, json, hashlib, subprocess, warnings
import numpy as np, pandas as pd, torch, h5py
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import roc_auc_score
import clip
from zero_shot import load_clip
from select_checkpoints import make_loader
from eval_nih import NIH_LABELS, LABEL_PROMPTS, build_groundtruth as nih_groundtruth

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

ROOT     = "/mnt/My_Doc/github/xray_clip/"
CHEXZERO = ROOT + "CheXzero/"
sys.path.insert(0, CHEXZERO)
os.chdir(CHEXZERO)

# ---- one deletable folder holds everything this notebook produces -------------------
EXP      = os.environ.get("F6_EXP_DIR", "/mnt/My_Doc/github/xray_clip/runs/freq_v1")
CKPT_DIR = os.path.join(EXP, "checkpoints", "pt-imp-f6")
OUT      = os.path.join(EXP, "results")
print("experiment directory:", EXP)
os.makedirs(OUT, exist_ok=True)

DEVICE     = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
BATCH, CTX = 256, 77
SEEDS      = 20
K_SWEEP    = [1, 2, 4, 8, 16, 32, 64]
N_SEL      = 6000          # NIH images reserved for checkpoint selection.
                           # NIH: 112,120
                           #    train_val_list.txt: 86,524                                                      
                            #    promptdev_list.txt: 20,000
                            #        ceckpoint selection :  6,000
                            #        few-shot support set: 14,000 
                           #    test_list.txt     : 25,596
SEED       = 0

# ---- inputs (all already on disk) ----------------------------------------------------
CXR_H5   = "/mnt/My_Doc/dataset/CXR_dataset/MIMIC_CXR/cxr_cache/cxr.h5"       # MIMIC, 377,110, all views. Created from 1_dataset_preparation.ipynb
TXT_CSV  = "data/mimic_impressions.csv"                                       # IMPRESSION ONLY - the paper's recipe
CXP_ROOT = "/mnt/My_Doc/dataset/CXR_dataset/CheXpert/"

CXP_TEST_IMG     = CXP_ROOT + "chexlocalize/chexlocalize/CheXpert/test"
CXP_TEST_LABELS  = CXP_ROOT + "chexlocalize/chexlocalize/CheXpert/test_labels.csv"
CXP_TRAIN_LABELS = CXP_ROOT + "kaggle/train.csv"
CXP_VAL_H5       = "data/chexpert_valid.h5"
CXP_VAL_PATHS    = "data/chexpert_valid_paths.csv"
CXP_VAL_CSV      = CXP_ROOT + "kaggle/valid.csv"

NIH_ENTRY                   = "/mnt/My_Doc/dataset/CXR_dataset/ChestXray14/cxr8/Data_Entry_2017_v2020.csv"
NIH_POOL_H5, NIH_POOL_PATHS = "data/nih_promptdev.h5", "data/nih_promptdev_paths.csv"
NIH_TEST_H5, NIH_TEST_PATHS = "data/nih_test.h5", "data/nih_test_paths.csv"

CXP_LABELS_5 = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]

# print(f"device     : {DEVICE}")
# print(f"experiment : {EXP}")
# print(f"results    : {OUT}")

experiment directory: /mnt/My_Doc/github/xray_clip/runs/freq_v1


## 1. Train the model

MIMIC-CXR, 377,110 image–report pairs, **impression only** — the recipe in the CheXzero paper.
8 epochs so that checkpoint selection has the full range to choose from; the paper's 4 epochs was
itself a hyperparameter they tuned on validation, not a constant.

The cell below **skips training if the checkpoints already exist.** Delete `EXP` to force a
retrain.

In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)
existing = sorted(f for f in os.listdir(CKPT_DIR) if f.endswith(".pt"))

if len(existing) >= 200:
    print(f"SKIP training -- {len(existing)} checkpoints already in {CKPT_DIR}")
else:
    # alignment gate: a silent mismatch here trains on wrong image/text pairs
    with h5py.File(CXR_H5, "r") as f:
        n_img = f["cxr"].shape[0]
    txt = pd.read_csv(TXT_CSV)
    assert n_img == len(txt), f"MISALIGNED: h5 {n_img:,} vs text {len(txt):,}"
    print(f"gate OK: {n_img:,} images aligned with {len(txt):,} impressions")
    print(f"         NO IMPRESSION {(txt.impression.astype(str)=='NO IMPRESSION').mean()*100:.1f}%")

    cmd = [sys.executable, "-u", "run_train.py",
           "--cxr_filepath", CXR_H5, "--txt_filepath", TXT_CSV,
           "--batch_size", "64", "--epochs", "8", "--lr", "1e-4",
           "--optimizer", "sgd", "--momentum", "0.9", "--save_interval", "200",
           "--save_dir", os.path.join(EXP, "checkpoints"), "--model_name", "pt-imp-f6"]
    print("\ntraining (~1h35m) ...", " ".join(cmd[-10:]))
    with open(os.path.join(EXP, "train.log"), "w") as log:
        subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, check=True)
    existing = sorted(f for f in os.listdir(CKPT_DIR) if f.endswith(".pt"))
    print(f"done -- {len(existing)} checkpoints")

CKPT_PATHS = sorted(os.path.join(CKPT_DIR, f) for f in os.listdir(CKPT_DIR) if f.endswith(".pt"))
print(f"\n{len(CKPT_PATHS)} checkpoints available")

## 2. How often is each pathology named in the training reports?

This is the independent variable for both laws. We count over the **same text the model was
trained on** — `mimic_impressions.csv`, impression sections only.

Three refinements matter:

* **Synonyms.** "cardiomegaly" is also written "enlarged cardiac silhouette". Counting the
  literal label name alone badly undercounts.
* **Negation.** Most mentions of pneumothorax are *denials* — "no pneumothorax". A denial
  teaches the model the word, but not what the finding looks like. We subtract them.
* **Affirmative rate** is what we regress on: mentions that assert the finding is present.

In [ ]:
reports = pd.read_csv(TXT_CSV)["impression"].fillna("").str.lower()
N_REPORTS = len(reports)

SYNONYMS = {
 "Atelectasis":        ["atelecta", "collapse of the", "lung collapse"],
 "Cardiomegaly":       ["cardiomegaly", "enlarged heart", "enlargement of the heart",
                        "cardiac enlargement", "enlarged cardiac silhouette",
                        "heart size is enlarged", "cardiac silhouette is enlarged"],
 "Effusion":           ["effusion"],
 "Pleural Effusion":   ["effusion"],
 "Infiltration":       ["infiltrat"],
 "Mass":               [r"\bmass\b", r"\bmasses\b"],
 "Nodule":             ["nodul"],
 "Pneumonia":          ["pneumonia", "infectious consolidation"],
 "Pneumothorax":       ["pneumothorax", "pneumothoraces"],
 "Consolidation":      ["consolidat"],
 "Edema":              [r"\bedema\b", "pulmonary edema", "fluid overload"],
 "Emphysema":          ["emphysema", "emphysematous", "hyperinflat"],
 "Fibrosis":           ["fibrosis", "fibrotic", "reticular"],
 "Pleural_Thickening": ["pleural thickening", "thickened pleura", "pleural scarring"],
 "Hernia":             ["hernia"],
 # CheXpert-only categories, for the 13-label test in section 10
 "Enlarged Cardiomediastinum": ["mediastinal widening", "widened mediastinum",
                                "enlarged cardiomediastinal", "cardiomediastinal silhouette is enlarg"],
 "Lung Opacity":       ["opacit"],
 "Lung Lesion":        [r"\blesion\b", "nodul", r"\bmass\b"],
 "Pleural Other":      ["pleural thickening", "pleural scarring", "pleural plaque"],
 "Fracture":           ["fractur"],
}

# a negation cue within 40 characters before the mention, no sentence break in between
NEG = re.compile(r"\b(no|not|without|free of|absence of|negative for|rule out|ruled out|"
                 r"resolved|clear of|unremarkable for)\b[^.;]{0,40}$")

def mention_stats(label):
    pat = re.compile("|".join(SYNONYMS[label]))
    hit = reports.str.contains(pat, regex=True)
    n   = int(hit.sum())
    neg = sum(any(NEG.search(s[:m.start()]) for m in pat.finditer(s)) for s in reports[hit])
    return {"n_mentions": n, "negated_%": 100*neg/max(n,1),
            "affirmative_n": n-neg, "affirmative_%": 100*(n-neg)/N_REPORTS}

FREQ_CACHE = os.path.join(OUT, "mention_rates.csv")
if os.path.exists(FREQ_CACHE):
    freq_all = pd.read_csv(FREQ_CACHE, index_col=0)
    print(f"cache hit: {FREQ_CACHE}")
else:
    freq_all = pd.DataFrame({l: mention_stats(l) for l in SYNONYMS}).T
    freq_all.to_csv(FREQ_CACHE)

freq_all["log_f"] = np.log10(freq_all["affirmative_%"])
print(f"reports counted: {N_REPORTS:,}\n")
print(freq_all.loc[NIH_LABELS, ["n_mentions","negated_%","affirmative_n","affirmative_%"]]
      .sort_values("affirmative_%", ascending=False).round(2).to_string())

### Negation is the reason counting words is not enough

A high mention count with a high negation rate means the model saw the *word* often but almost
never saw the *finding*. Pneumothorax is the extreme case.

In [ ]:
neg_view = freq_all.loc[NIH_LABELS, ["n_mentions","negated_%","affirmative_%"]].copy()
neg_view["mentions_%"] = 100*neg_view.n_mentions/N_REPORTS
print(neg_view[["mentions_%","negated_%","affirmative_%"]]
      .sort_values("negated_%", ascending=False).round(2).to_string())
print("\nPneumothorax: the word is common, the finding is not -- "
      f"{neg_view.loc['Pneumothorax','negated_%']:.0f}% of its mentions are denials.")

## 3. Data splits

Each dataset gets three disjoint roles. **Nothing is reused across roles**, and nothing from a
test set is ever seen before scoring.

| | selection | support (few-shot) | test |
|---|---|---|---|
| **CheXpert** | val, 200 studies | 10,000 from CheXpert train | 500 studies |
| **ChestX-ray14** | 6,000 from promptdev | the other 14,000 | 25,596 images |

The NIH selection and support sets are carved from the same 20,000-image pool by a fixed
permutation, so they are disjoint by construction. Every split is asserted below.

In [ ]:
# ---------- CheXpert test: 500 frontal images, one per study ------------------------
CXP_TEST_H5    = os.path.join(EXP, "chexpert_test.h5")
CXP_TEST_PATHS = os.path.join(EXP, "chexpert_test_paths.csv")
if not os.path.exists(CXP_TEST_H5):
    print("preprocessing CheXpert test ...")
    subprocess.run([sys.executable, "-u", "run_preprocess.py",
        "--dataset_type", "chexpert-valid",          # same view1 filter, and writes the paths csv
        "--chest_x_ray_path", CXP_TEST_IMG,
        "--csv_out_path", CXP_TEST_PATHS, "--cxr_out_path", CXP_TEST_H5],
        stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, check=True)

# ---------- CheXpert support pool: 10,000 frontal training images --------------------
CXP_SUPP_H5    = os.path.join(EXP, "chexpert_support.h5")
CXP_SUPP_PATHS = os.path.join(EXP, "chexpert_support_paths.csv")
if not os.path.exists(CXP_SUPP_H5):
    print("building CheXpert support pool ...")
    from data_process import img_to_hdf5
    tr = pd.read_csv(CXP_TRAIN_LABELS)
    tr = tr[tr["Frontal/Lateral"] == "Frontal"].copy()
    tr["rel"] = tr.Path.str.extract(r"(train/patient\d+/study\d+/view\d+_\w+\.jpg)")[0]
    tr = tr.dropna(subset=["rel"])
    rng = np.random.default_rng(SEED)
    cand = [f"{CXP_ROOT}kaggle/{r}" for r in tr.iloc[rng.permutation(len(tr))].rel.head(20000)]
    paths = [p for p in cand if os.path.exists(p)][:10000]
    pd.DataFrame({"Path": paths}).to_csv(CXP_SUPP_PATHS, index=False)
    img_to_hdf5(paths, CXP_SUPP_H5)

# ---------- NIH: split the 20,000-image pool into selection + support ----------------
nih_pool_paths = pd.read_csv(NIH_POOL_PATHS)["Path"]
n_pool = len(nih_pool_paths)
perm = np.random.default_rng(SEED).permutation(n_pool)
NIH_SEL_IDX, NIH_SUPP_IDX = np.sort(perm[:N_SEL]), np.sort(perm[N_SEL:])
assert not set(NIH_SEL_IDX) & set(NIH_SUPP_IDX)
print(f"NIH pool {n_pool:,}  ->  selection {len(NIH_SEL_IDX):,}  +  support {len(NIH_SUPP_IDX):,}")

# ---------- ground truth, each aligned to its h5 row order ---------------------------
study_key = lambda s: s.str.extract(r"(patient\d+/study\d+)")[0]

tl = pd.read_csv(CXP_TEST_LABELS)
tl["study"] = study_key(tl.Path)
assert (tl.groupby("study")[CXP_LABELS_5].nunique() <= 1).all().all(), "labels differ within a study"
tl = tl.drop_duplicates("study").set_index("study")        # 668 image rows -> 500 study rows
cxp_test_order = study_key(pd.read_csv(CXP_TEST_PATHS)["Path"])
assert len(cxp_test_order) == len(set(cxp_test_order)) == 500
y_cxp_test_full = tl.loc[cxp_test_order]                   # keep all columns for section 10

PAT = r"(train/patient\d+/study\d+/view\d+_\w+\.jpg)"
trl = pd.read_csv(CXP_TRAIN_LABELS)
trl["k"] = trl.Path.str.extract(PAT)[0]
trl = trl.dropna(subset=["k"]).drop_duplicates("k").set_index("k")
cxp_supp_key = pd.read_csv(CXP_SUPP_PATHS)["Path"].str.extract(PAT)[0]
assert cxp_supp_key.notna().all() and cxp_supp_key.isin(trl.index).all()
y_cxp_supp_full = trl.loc[cxp_supp_key]

cxp_val_order = pd.read_csv(CXP_VAL_PATHS)["Path"].str.extract(r"(patient\d+/study\d+)")[0]
vl = pd.read_csv(CXP_VAL_CSV)
vl["study"] = study_key(vl.Path)
vl = vl.drop_duplicates("study").set_index("study")
y_cxp_val = vl.loc[cxp_val_order, CXP_LABELS_5].to_numpy(float)

y_nih_test = nih_groundtruth(NIH_ENTRY, NIH_TEST_PATHS, list(NIH_LABELS))
y_nih_pool = nih_groundtruth(NIH_ENTRY, NIH_POOL_PATHS, list(NIH_LABELS))

# ---------- shape + leakage assertions ------------------------------------------------
for name, h5, n_lab in [("CheXpert test", CXP_TEST_H5, len(y_cxp_test_full)),
                        ("CheXpert supp", CXP_SUPP_H5, len(y_cxp_supp_full)),
                        ("CheXpert val",  CXP_VAL_H5,  len(y_cxp_val)),
                        ("NIH test",      NIH_TEST_H5, len(y_nih_test)),
                        ("NIH pool",      NIH_POOL_H5, len(y_nih_pool))]:
    with h5py.File(h5, "r") as f: n = f["cxr"].shape[0]
    assert n == n_lab, f"{name}: h5 {n} vs labels {n_lab}"
    print(f"  {name:<14} {n:>7,} rows  aligned")

full = lambda p: set(pd.read_csv(p)["Path"])
pid  = lambda p: set(pd.read_csv(p)["Path"].str.extract(r"(patient\d+)")[0].dropna())
# CheXpert files are ALL named view1_frontal.jpg, so basenames collide meaninglessly --
# compare full paths, and patient ids, never basenames.
print(f"\n  CheXpert test/support  paths {len(full(CXP_TEST_PATHS)&full(CXP_SUPP_PATHS))}  "
      f"patients {len(pid(CXP_TEST_PATHS)&pid(CXP_SUPP_PATHS))}")
print(f"  NIH test/pool          images {len(full(NIH_TEST_PATHS)&full(NIH_POOL_PATHS))}")

## 4. Checkpoint selection — each dataset picks its own

Every checkpoint is scored on its dataset's **selection** split, and the single best is kept.
This is where the earlier run went wrong: it ranked once on CheXpert validation and reused that
ranking for ChestX-ray14.

Scoring uses the paper's prompt frame, `"{}"` versus `"no {}"`, with no tuning.

In [ ]:
CXR_PAIR = ("{}", "no {}")

def text_vecs(model, strings):
    with torch.no_grad():
        e = model.encode_text(clip.tokenize(list(strings), context_length=CTX).to(DEVICE))
        return (e / e.norm(dim=-1, keepdim=True)).float()

def score_checkpoint(ckpt, loader, labels, prompt_map, rows=None):
    """(n_images, n_labels) softmax probabilities for one checkpoint."""
    m = load_clip(model_path=ckpt, pretrained=True, context_length=CTX).to(DEVICE).eval()
    pos = text_vecs(m, [CXR_PAIR[0].format(prompt_map[l]) for l in labels])
    neg = text_vecs(m, [CXR_PAIR[1].format(prompt_map[l]) for l in labels])
    chunks = []
    with torch.no_grad():
        for b in loader:
            f = m.encode_image(b["img"].to(DEVICE))
            f = (f / f.norm(dim=-1, keepdim=True)).float()
            chunks.append(torch.sigmoid(f @ pos.T - f @ neg.T).cpu())
    del m
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    p = torch.cat(chunks).numpy()
    return p if rows is None else p[rows]

def rank_checkpoints(tag, h5, labels, prompt_map, y, rows=None):
    cache = os.path.join(OUT, f"ranking_{tag}.csv")
    if os.path.exists(cache):
        r = pd.read_csv(cache); print(f"{tag}: cache hit ({len(r)} checkpoints)"); return r
    loader = make_loader(h5, BATCH)
    out = []
    for i, c in enumerate(CKPT_PATHS, 1):
        p = score_checkpoint(c, loader, labels, prompt_map, rows)
        aucs = [roc_auc_score(y[:, j], p[:, j]) for j in range(len(labels))]
        out.append({"checkpoint": c, "step": int(re.search(r"_(-?\d+)\.pt", c).group(1)),
                    "mean_auc": float(np.mean(aucs))})
        if i % 40 == 0 or i == len(CKPT_PATHS):
            print(f"  {tag} [{i}/{len(CKPT_PATHS)}] best so far "
                  f"{max(o['mean_auc'] for o in out):.4f}", flush=True)
    r = pd.DataFrame(out).sort_values("mean_auc", ascending=False).reset_index(drop=True)
    r.to_csv(cache, index=False)
    return r

CXP_PROMPT = {l: l for l in CXP_LABELS_5}
NIH_PROMPT = {l: LABEL_PROMPTS.get(l, l) for l in NIH_LABELS}

print("ranking on CheXpert val (200 studies) ...")
rank_cxp = rank_checkpoints("chexpert", CXP_VAL_H5, CXP_LABELS_5, CXP_PROMPT, y_cxp_val)
print("\nranking on held-out NIH split (6,000 images) ...")
rank_nih = rank_checkpoints("nih", NIH_POOL_H5, list(NIH_LABELS), NIH_PROMPT,
                            y_nih_pool[NIH_SEL_IDX], rows=NIH_SEL_IDX)

BEST_CXP, BEST_NIH = rank_cxp.iloc[0], rank_nih.iloc[0]
SPE = 5893
print(f"\nCheXpert  best = step {BEST_CXP.step:>6}  (epoch {BEST_CXP.step/SPE:.2f})  "
      f"selection AUC {BEST_CXP.mean_auc:.4f}")
print(f"NIH       best = step {BEST_NIH.step:>6}  (epoch {BEST_NIH.step/SPE:.2f})  "
      f"selection AUC {BEST_NIH.mean_auc:.4f}")
print(f"same checkpoint chosen by both? {BEST_CXP.checkpoint == BEST_NIH.checkpoint}")

### Where does the optimum sit, epoch by epoch?

The paper trains 4 epochs. That was a hyperparameter they tuned on their own validation runs,
not a constant — so it is worth checking where ours actually peaks.

In [ ]:
for tag, r in [("CheXpert", rank_cxp), ("NIH", rank_nih)]:
    g = r.assign(ep=np.ceil(r.step.clip(lower=0)/SPE).astype(int).clip(1, 8)).groupby("ep")
    print(f"\n{tag}:")
    for ep, grp in g:
        b = grp.loc[grp.mean_auc.idxmax()]
        print(f"  epoch {ep}: best step {int(b.step):>6}  AUC {b.mean_auc:.4f}   "
              f"(epoch mean {grp.mean_auc.mean():.4f})")

## 5. Encode once, with each dataset's own checkpoint

Image features do not depend on K, on the seed, or on the prompt. Encode each set once and
everything afterwards is arithmetic on cached vectors.

In [ ]:
def encode(h5, ckpt, tag, rows=None):
    key = hashlib.md5((ckpt + h5 + str(rows is not None)).encode()).hexdigest()[:10]
    cache = os.path.join(OUT, f"feat_{tag}_{key}.npy")
    if os.path.exists(cache):
        a = np.load(cache); print(f"  {tag}: cache hit {a.shape}"); return torch.from_numpy(a)
    m = load_clip(model_path=ckpt, pretrained=True, context_length=CTX).to(DEVICE).eval()
    chunks = []
    with torch.no_grad():
        for b in make_loader(h5, BATCH):
            f = m.encode_image(b["img"].to(DEVICE))
            chunks.append((f / f.norm(dim=-1, keepdim=True)).float().cpu())
    del m
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    a = torch.cat(chunks).numpy()
    if rows is not None: a = a[rows]
    np.save(cache, a); print(f"  {tag}: encoded {a.shape}")
    return torch.from_numpy(a)

def text_cache(ckpt, strings):
    m = load_clip(model_path=ckpt, pretrained=True, context_length=CTX).to(DEVICE).eval()
    v = text_vecs(m, strings).cpu()
    del m
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return {s: v[i] for i, s in enumerate(strings)}

DATA = {}
for name, ckpt, test_h5, supp_h5, y_test, y_supp, labels, pmap, supp_rows in [
    ("CheXpert", BEST_CXP.checkpoint, CXP_TEST_H5, CXP_SUPP_H5,
     y_cxp_test_full[CXP_LABELS_5].to_numpy(float), y_cxp_supp_full[CXP_LABELS_5].to_numpy(float),
     CXP_LABELS_5, CXP_PROMPT, None),
    ("NIH", BEST_NIH.checkpoint, NIH_TEST_H5, NIH_POOL_H5,
     y_nih_test, y_nih_pool[NIH_SUPP_IDX], list(NIH_LABELS), NIH_PROMPT, NIH_SUPP_IDX)]:
    print(f"{name}  (checkpoint {os.path.basename(ckpt)})")
    strings = sorted({t.format(pmap[l]) for l in labels for t in CXR_PAIR})
    DATA[name] = dict(ckpt=ckpt, labels=labels, pmap=pmap,
                      y_test=y_test, y_supp=y_supp,
                      F_test=encode(test_h5, ckpt, f"{name}_test"),
                      F_supp=encode(supp_h5, ckpt, f"{name}_supp", supp_rows),
                      T=text_cache(ckpt, strings))
    assert len(DATA[name]["F_test"]) == len(y_test)
    assert len(DATA[name]["F_supp"]) == len(y_supp)
print("\nall features cached")

## 6. Zero-shot

$$p(\text{label}) = \frac{e^{\,v\cdot t_{pos}}}{e^{\,v\cdot t_{pos}} + e^{\,v\cdot t_{neg}}}$$

A **single** checkpoint — no ensemble. The reference point is therefore CheXzero's single-model
figure, **0.878**, not their ensembled 0.889.

In [ ]:
def pair_prob(F, w_pos, w_neg):
    """sigmoid(a-b) equals the 2-way softmax, but is numerically stable."""
    return torch.sigmoid(F @ w_pos - F @ w_neg).numpy()

for name, d in DATA.items():
    d["zs_prob"] = {l: pair_prob(d["F_test"], d["T"][CXR_PAIR[0].format(d["pmap"][l])],
                                              d["T"][CXR_PAIR[1].format(d["pmap"][l])])
                    for l in d["labels"]}
    d["zs_auc"] = {l: roc_auc_score(d["y_test"][:, i], d["zs_prob"][l])
                   for i, l in enumerate(d["labels"])}
    d["ZS"] = float(np.mean(list(d["zs_auc"].values())))

for name, d in DATA.items():
    print(f"\n=== {name} zero-shot, {len(d['labels'])} labels ===")
    for l in d["labels"]:
        print(f"  {l:<22}{d['zs_auc'][l]:>8.4f}")
    print(f"  {'MEAN':<22}{d['ZS']:>8.4f}")

print("\n" + "="*54)
print(f"CheXpert  ours {DATA['CheXpert']['ZS']:.4f}   CheXzero single model 0.878   "
      f"diff {DATA['CheXpert']['ZS']-0.878:+.4f}")
print(f"NIH-14    ours {DATA['NIH']['ZS']:.4f}")
print("="*54)

In [ ]:
def boot_mean_auc(y, probs, labels, n_boot=1000, seed=0):
    rng, n, out = np.random.default_rng(seed), len(y), []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        a = [roc_auc_score(y[idx, i], probs[l][idx]) for i, l in enumerate(labels)
             if y[idx, i].min() != y[idx, i].max()]
        if a: out.append(np.mean(a))
    d = np.array(out)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

for name, d in DATA.items():
    m, lo, hi = boot_mean_auc(d["y_test"], d["zs_prob"], d["labels"])
    note = f"   0.878 inside CI: {'yes' if lo <= 0.878 <= hi else 'no'}" if name == "CheXpert" else ""
    print(f"{name:<9} {m:.4f}  95% CI [{lo:.4f}, {hi:.4f}]{note}")

## 7. Law 1 — frequency predicts zero-shot accuracy

The replication step. Regress test AUC on $\log_{10}$ of the affirmative mention rate.

Fourteen points is few, so three guards: report Spearman alongside Pearson, test the rare/common
split non-parametrically, and check the obvious confound — that rare pathologies simply have
fewer test positives, which would make their AUC noisier rather than genuinely worse.

In [ ]:
law1 = freq_all.loc[NIH_LABELS, ["affirmative_%", "log_f"]].copy()
law1["auc"]   = pd.Series(DATA["NIH"]["zs_auc"])
law1["n_pos"] = y_nih_test.sum(0).astype(int)

r, p   = stats.pearsonr(law1.log_f, law1.auc)
rho, ps = stats.spearmanr(law1.log_f, law1.auc)
a1, b1 = np.polyfit(law1.log_f, law1.auc, 1)

print(f"LAW 1   AUC = {a1:+.4f} * log10(f) + {b1:.4f}      n = {len(law1)}")
print(f"        pearson  r = {r:+.3f}  p = {p:.4f}")
print(f"        spearman rho = {rho:+.3f}  p = {ps:.4f}")
print(f"        -> 10x more affirmative mentions buys {a1:+.4f} AUC\n")

med = law1["affirmative_%"].median()
lo_g, hi_g = law1[law1["affirmative_%"] < med], law1[law1["affirmative_%"] >= med]
u, pu = stats.mannwhitneyu(hi_g.auc, lo_g.auc, alternative="greater")
print(f"RARE   n={len(lo_g)}  mean mentions {lo_g['affirmative_%'].mean():5.2f}%  mean AUC {lo_g.auc.mean():.4f}")
print(f"COMMON n={len(hi_g)}  mean mentions {hi_g['affirmative_%'].mean():5.2f}%  mean AUC {hi_g.auc.mean():.4f}")
print(f"gap = {hi_g.auc.mean()-lo_g.auc.mean():+.4f}   Mann-Whitney p = {pu:.4f}\n")

rn, pn = stats.pearsonr(np.log10(law1.n_pos), law1.auc)
res_auc = law1.auc - np.poly1d(np.polyfit(np.log10(law1.n_pos), law1.auc, 1))(np.log10(law1.n_pos))
res_f   = law1.log_f - np.poly1d(np.polyfit(np.log10(law1.n_pos), law1.log_f, 1))(np.log10(law1.n_pos))
rp, pp  = stats.pearsonr(res_f, res_auc)
print(f"CONFOUND  test positives vs AUC:  r = {rn:+.3f}  p = {pn:.4f}")
print(f"          partial r controlling for it: {rp:+.3f}  p = {pp:.4f}   "
      f"-> {'survives' if pp < 0.05 else 'DOES NOT survive'}")

## 8. Few-shot — SS-Text

$$w_{pos} = \sum_{i \in \text{pos}} v_i + t_{pos}, \qquad w_{neg} = \sum_{i \in \text{neg}} v_i + t_{neg}$$

Then score exactly as before. Nothing in the encoder is trained — one vector addition per class.

**`K = 0` must reproduce zero-shot exactly**: the sums vanish and only the text prototypes
remain. That is asserted below; if it fails, every later number is suspect.

Note this is a **sum, not an average** — so the K images outweigh the single text vector roughly
K to 1, and by K=16 the prompt contributes about 6% of the classifier. Few-shot does not refine
the prompt, it replaces it.

In [ ]:
def ss_text(d, K, rng):
    probs = {}
    for i, l in enumerate(d["labels"]):
        t_pos = d["T"][CXR_PAIR[0].format(d["pmap"][l])]
        t_neg = d["T"][CXR_PAIR[1].format(d["pmap"][l])]
        c = d["y_supp"][:, i]
        # CheXpert convention: blank means "not mentioned", which the official evaluation and
        # test_labels.csv both treat as negative. -1 (uncertain) is excluded from both pools.
        # NIH has neither blanks nor -1, so this reduces to the obvious rule there.
        pool_p = np.flatnonzero(c == 1)
        pool_n = np.flatnonzero((c == 0) | np.isnan(c))
        k = min(K, len(pool_p), len(pool_n))
        if k == 0:
            probs[l] = pair_prob(d["F_test"], t_pos, t_neg); continue
        ip  = rng.choice(pool_p, k, replace=False)
        inn = rng.choice(pool_n, k, replace=False)
        probs[l] = pair_prob(d["F_test"],
                             d["F_supp"][ip].sum(0) + t_pos,
                             d["F_supp"][inn].sum(0) + t_neg)
    return probs

for name, d in DATA.items():
    p0 = ss_text(d, 0, np.random.default_rng(0))
    diff = max(abs(p0[l] - d["zs_prob"][l]).max() for l in d["labels"])
    print(f"{name:<9} K=0 reproduces zero-shot: max abs diff {diff:.2e}   "
          f"{'OK' if diff < 1e-9 else 'MISMATCH - STOP'}")

In [ ]:
SWEEP_CACHE = os.path.join(OUT, "fewshot_sweep.csv")
if os.path.exists(SWEEP_CACHE):
    fs = pd.read_csv(SWEEP_CACHE); print(f"cache hit: {len(fs):,} rows")
else:
    rows = []
    for name, d in DATA.items():
        for K in K_SWEEP:
            for s in range(SEEDS):
                pr = ss_text(d, K, np.random.default_rng(s))
                for i, l in enumerate(d["labels"]):
                    rows.append({"dataset": name, "K": K, "seed": s, "label": l,
                                 "auc": roc_auc_score(d["y_test"][:, i], pr[l])})
            print(f"  {name:<9} K={K:<3} done", flush=True)
    fs = pd.DataFrame(rows); fs.to_csv(SWEEP_CACHE, index=False)
    print(f"\n{len(fs):,} rows -> {SWEEP_CACHE}")

summary = (fs.groupby(["dataset","K","seed"])["auc"].mean().reset_index()
             .groupby(["dataset","K"])["auc"].agg(["mean","std"]).reset_index())
for name, d in DATA.items():
    s = summary[summary.dataset == name]
    print(f"\n=== {name} ({len(d['labels'])} labels) ===")
    print(f"{'K':>4}{'mean AUC':>11}{'std':>8}{'vs zero-shot':>14}")
    print(f"{0:>4}{d['ZS']:>11.4f}{'-':>8}{'-':>14}")
    for _, r in s.iterrows():
        print(f"{int(r.K):>4}{r['mean']:>11.4f}{r['std']:>8.4f}{r['mean']-d['ZS']:>+14.4f}")

## 9. Law 2 — frequency predicts the *benefit* of adaptation

**This is the contribution.** Same independent variable as Law 1, new dependent variable: the
change in AUC from few-shot.

If the sign of the slope is negative, then the measurement that predicts good zero-shot
performance also predicts that adaptation will *hurt*.

In [ ]:
def delta_table(name, K):
    d = DATA[name]
    m = fs[(fs.dataset == name) & (fs.K == K)].groupby("label")["auc"].mean()
    t = pd.DataFrame({"zero_shot": pd.Series(d["zs_auc"]), "few_shot": m})
    t["delta"] = t.few_shot - t.zero_shot
    t["affirmative_%"] = freq_all.loc[t.index, "affirmative_%"]
    t["log_f"] = freq_all.loc[t.index, "log_f"]
    return t.loc[d["labels"]]

law2 = {}
print(f"{'dataset':<10}{'K':>4}{'pearson r':>12}{'p':>9}{'spearman':>11}{'p':>9}{'slope':>10}")
print("-" * 66)
for name in DATA:
    for K in K_SWEEP:
        t = delta_table(name, K)
        if len(t) < 5: continue
        r, p   = stats.pearsonr(t.log_f, t.delta)
        rho, ps = stats.spearmanr(t.log_f, t.delta)
        sl, ic = np.polyfit(t.log_f, t.delta, 1)
        law2[(name, K)] = dict(r=r, p=p, rho=rho, ps=ps, slope=sl, intercept=ic, table=t)
        star = " *" if min(p, ps) < 0.05 else ""
        print(f"{name:<10}{K:>4}{r:>12.3f}{p:>9.4f}{rho:>11.3f}{ps:>9.4f}{sl:>10.4f}{star}")
print("\n* = significant at 0.05 on Pearson or Spearman")

In [ ]:
K_REF = 64
t = law2[("NIH", K_REF)]["table"].sort_values("delta", ascending=False)
st = law2[("NIH", K_REF)]
print(f"ChestX-ray14, K={K_REF}  —  ordered by benefit\n")
print(f"{'label':<22}{'mentions %':>12}{'zero-shot':>11}{'few-shot':>10}{'delta':>10}")
print("-" * 65)
for l, r_ in t.iterrows():
    print(f"{l:<22}{r_['affirmative_%']:>12.2f}{r_.zero_shot:>11.4f}{r_.few_shot:>10.4f}{r_.delta:>+10.4f}")
print("-" * 65)
print(f"{'improved':<22}{int((t.delta>0).sum())} of {len(t)}")
print(f"\nLAW 2   dAUC = {st['slope']:+.4f} * log10(f) + {st['intercept']:.4f}")
print(f"        pearson r = {st['r']:+.3f} (p={st['p']:.4f})   "
      f"spearman rho = {st['rho']:+.3f} (p={st['ps']:.4f})")

### Does frequency predict the *sign*?

The regression could be driven by one outlier. A non-parametric check: split at the median
mention rate and compare the two groups' benefits directly.

In [ ]:
print(f"{'dataset':<10}{'K':>4}{'rare helped':>14}{'common helped':>16}"
      f"{'rare mean':>12}{'common mean':>13}{'p':>9}")
print("-" * 80)
for name in DATA:
    for K in [16, 64]:
        t = law2[(name, K)]["table"]
        med = t["affirmative_%"].median()
        lo_g, hi_g = t[t["affirmative_%"] < med], t[t["affirmative_%"] >= med]
        try:
            _, pv = stats.mannwhitneyu(lo_g.delta, hi_g.delta, alternative="greater")
        except ValueError:
            pv = float("nan")
        print(f"{name:<10}{K:>4}{f'{int((lo_g.delta>0).sum())}/{len(lo_g)}':>14}"
              f"{f'{int((hi_g.delta>0).sum())}/{len(hi_g)}':>16}"
              f"{lo_g.delta.mean():>+12.4f}{hi_g.delta.mean():>+13.4f}{pv:>9.4f}")

## 10. The sign flip, and the crossing point

Put the two laws on one axis. Law 1 rises with frequency; Law 2 falls. Where Law 2 crosses zero
is the mention rate above which adaptation stops paying for itself — a rule that uses only the
training corpus, never test data.

In [ ]:
st = law2[("NIH", K_REF)]
cross = 10 ** (-st["intercept"] / st["slope"])
print(f"LAW 1   zero-shot AUC   = {a1:+.4f} * log10(f) + {b1:.4f}")
print(f"LAW 2   few-shot delta  = {st['slope']:+.4f} * log10(f) + {st['intercept']:.4f}")
print(f"\nLaw 2 crosses zero at f = {cross:.2f}% affirmative mentions")
print(f"-> adapt a pathology only if it is named in fewer than {cross:.1f}% of reports\n")
side = law2[("NIH", K_REF)]["table"].assign(
    predicted=lambda x: np.where(x["affirmative_%"] < cross, "adapt", "keep zero-shot"),
    actual=lambda x: np.where(x.delta > 0, "adapt", "keep zero-shot"))
agree = (side.predicted == side.actual).sum()
print(f"rule agrees with the outcome on {agree}/{len(side)} pathologies")
print(side[["affirmative_%", "delta", "predicted", "actual"]].round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
xs = np.linspace(law1.log_f.min()-0.1, law1.log_f.max()+0.1, 50)

ax = axes[0]
ax.scatter(law1.log_f, law1.auc, s=70, color="#065A82", zorder=3)
ax.plot(xs, a1*xs + b1, color="#065A82", lw=1.5)
for l, r_ in law1.iterrows():
    ax.annotate(l[:11], (r_.log_f, r_.auc), fontsize=7.5, xytext=(3,3), textcoords="offset points")
ax.set_xlabel("log10(affirmative mention rate, %)"); ax.set_ylabel("zero-shot AUC")
ax.set_title(f"Law 1: slope {a1:+.3f}  (r={r:+.2f}, p={p:.3f})"); ax.grid(alpha=.3)

ax = axes[1]
t = st["table"]
ax.scatter(t.log_f, t.delta, s=70, color="#B85042", zorder=3)
ax.plot(xs, st["slope"]*xs + st["intercept"], color="#B85042", lw=1.5)
ax.axhline(0, color="k", lw=.8)
ax.axvline(np.log10(cross), color="#02A37A", ls="--", lw=1.4, label=f"crossing  f={cross:.1f}%")
for l, r_ in t.iterrows():
    ax.annotate(l[:11], (r_.log_f, r_.delta), fontsize=7.5, xytext=(3,3), textcoords="offset points")
ax.set_xlabel("log10(affirmative mention rate, %)"); ax.set_ylabel(f"AUC change at K={K_REF}")
ax.set_title(f"Law 2: slope {st['slope']:+.3f}  (rho={st['rho']:+.2f}, p={st['ps']:.3f})")
ax.grid(alpha=.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "two_laws.png"), dpi=150); plt.show()

## 11. Selective adaptation — what the rule is worth

Three policies compared:

| | |
|---|---|
| **uniform** | adapt every pathology, as SS-Text does today |
| **rule** | adapt only where mentions < the crossing point — uses the training corpus only |
| **oracle** | adapt only where it actually helped — uses **test labels**, so an upper bound, not achievable |

The gap between *rule* and *oracle* is how much the label-free rule leaves on the table.

In [ ]:
print(f"{'dataset':<10}{'K':>4}{'zero-shot':>11}{'uniform':>10}{'RULE':>10}{'oracle':>10}"
      f"{'rule gain':>11}")
print("-" * 68)
policy_rows = []
for name in DATA:
    zs = pd.Series(DATA[name]["zs_auc"])
    for K in [16, 64]:
        t = law2[(name, K)]["table"]
        uniform = t.few_shot.mean()
        rule    = np.where(t["affirmative_%"] < cross, t.few_shot, t.zero_shot).mean()
        oracle  = np.maximum(t.few_shot, t.zero_shot).mean()
        policy_rows.append({"dataset": name, "K": K, "zero_shot": zs.mean(),
                            "uniform": uniform, "rule": rule, "oracle": oracle})
        print(f"{name:<10}{K:>4}{zs.mean():>11.4f}{uniform:>10.4f}{rule:>10.4f}"
              f"{oracle:>10.4f}{rule-zs.mean():>+11.4f}")
pd.DataFrame(policy_rows).to_csv(os.path.join(OUT, "policies.csv"), index=False)

## 12. The falsification test — CheXpert's own rare pathologies

Every CheXpert label scored so far is **common** in MIMIC, and every one lost from adaptation.
A reviewer will reasonably object: *maybe CheXpert is simply a dataset where few-shot fails.*

CheXpert's test labels include genuinely rare findings. The hypothesis makes a sharp,
falsifiable prediction: **these should gain** — same dataset, same model, opposite sign.

| label | test positives |
|---|---|
| Fracture | 5 |
| Lung Lesion | 8 |
| Pneumothorax | 9 |
| Pneumonia | 11 |

With so few positives the AUCs are very noisy, so treat the direction as the signal and the
magnitudes as indicative. `Support Devices` and `No Finding` are excluded — they are not
pathologies.

In [ ]:
CXP13 = ["Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity", "Lung Lesion",
         "Edema", "Consolidation", "Pneumonia", "Atelectasis", "Pneumothorax",
         "Pleural Effusion", "Pleural Other", "Fracture"]
CXP13 = [l for l in CXP13 if l in freq_all.index and freq_all.loc[l, "affirmative_%"] > 0]

d = DATA["CheXpert"]
y13_test = y_cxp_test_full[CXP13].to_numpy(float)
y13_supp = y_cxp_supp_full[CXP13].to_numpy(float)
pmap13   = {l: l for l in CXP13}
strings13 = sorted({t.format(pmap13[l]) for l in CXP13 for t in CXR_PAIR})
T13 = text_cache(d["ckpt"], strings13)

d13 = dict(labels=CXP13, pmap=pmap13, T=T13, y_test=y13_test, y_supp=y13_supp,
           F_test=d["F_test"], F_supp=d["F_supp"])
d13["zs_prob"] = {l: pair_prob(d13["F_test"], T13[CXR_PAIR[0].format(l)],
                                              T13[CXR_PAIR[1].format(l)]) for l in CXP13}
d13["zs_auc"]  = {l: roc_auc_score(y13_test[:, i], d13["zs_prob"][l]) for i, l in enumerate(CXP13)}

rows = []
for K in [16, 64]:
    acc = {l: [] for l in CXP13}
    for s in range(SEEDS):
        pr = ss_text(d13, K, np.random.default_rng(s))
        for i, l in enumerate(CXP13):
            acc[l].append(roc_auc_score(y13_test[:, i], pr[l]))
    for l in CXP13:
        rows.append({"K": K, "label": l, "few_shot": np.mean(acc[l]),
                     "zero_shot": d13["zs_auc"][l],
                     "affirmative_%": freq_all.loc[l, "affirmative_%"],
                     "n_pos": int(y13_test[:, CXP13.index(l)].sum())})
cxp13 = pd.DataFrame(rows)
cxp13["delta"] = cxp13.few_shot - cxp13.zero_shot
cxp13.to_csv(os.path.join(OUT, "chexpert_13.csv"), index=False)

t64 = cxp13[cxp13.K == 64].sort_values("affirmative_%")
print(f"CheXpert, {len(CXP13)} pathologies, K=64  —  ordered by how rarely they are mentioned\n")
print(f"{'label':<30}{'mentions %':>12}{'n_pos':>7}{'zero-shot':>11}{'few-shot':>10}{'delta':>10}")
print("-" * 82)
for _, r_ in t64.iterrows():
    print(f"{r_.label:<30}{r_['affirmative_%']:>12.2f}{r_.n_pos:>7}{r_.zero_shot:>11.4f}"
          f"{r_.few_shot:>10.4f}{r_.delta:>+10.4f}")
print("-" * 82)
print(f"{'improved':<30}{int((t64.delta>0).sum())} of {len(t64)}")

rr, pr_ = stats.pearsonr(np.log10(t64["affirmative_%"]), t64.delta)
rr2, pr2 = stats.spearmanr(np.log10(t64["affirmative_%"]), t64.delta)
print(f"\nLaw 2 on CheXpert alone:  pearson r = {rr:+.3f} (p={pr_:.4f})   "
      f"spearman rho = {rr2:+.3f} (p={pr2:.4f})")
print("-> same sign as ChestX-ray14?  " +
      ("YES - the hypothesis survives on a second dataset"
       if rr < 0 else "NO - the hypothesis fails here"))

## 13. Everything in one table

The paper's evidence chain, end to end.

In [ ]:
final = pd.DataFrame([
    {"step": "1. Reproduction",
     "claim": "model matches the published single-model figure",
     "evidence": f"CheXpert test {DATA['CheXpert']['ZS']:.4f} vs CheXzero 0.878"},
    {"step": "2. Law 1 (replication)",
     "claim": "rarely-named pathologies score worse zero-shot",
     "evidence": f"slope {a1:+.4f}/decade, r={r:+.2f}, p={p:.3f}; "
                 f"partial r={rp:+.2f}, p={pp:.3f} controlling for test prevalence"},
    {"step": "3. Law 2 (new)",
     "claim": "rarely-named pathologies benefit MORE from adaptation",
     "evidence": f"slope {st['slope']:+.4f}/decade, rho={st['rho']:+.2f}, p={st['ps']:.4f} (NIH, K={K_REF})"},
    {"step": "4. Sign flip",
     "claim": "one measurement, opposite signs; the lines cross",
     "evidence": f"Law 1 {a1:+.3f} vs Law 2 {st['slope']:+.3f}; crossing at f={cross:.1f}%"},
    {"step": "5. Decision rule",
     "claim": "a label-free rule beats adapting everything",
     "evidence": "see section 11"},
    {"step": "6. Falsification",
     "claim": "holds on CheXpert's own rare pathologies",
     "evidence": f"Law 2 on CheXpert: rho={rr2:+.2f}, p={pr2:.4f}"},
])
pd.set_option("display.max_colwidth", 95)
display(final.style.hide(axis="index"))
final.to_csv(os.path.join(OUT, "evidence_chain.csv"), index=False)

## 14. Limitations — state every one of these in the paper

1. **n = 14 pathologies.** Small for a regression. Spearman is reported alongside Pearson
   throughout, and the rare/common split is tested non-parametrically, but the sample is the
   sample. PadChest offers 57 findings with n > 50 and is the obvious way to fix this.

2. **The crossing point is fitted on these points.** The ~7% threshold is an estimate from this
   data, not an independently validated constant. The CheXpert section is the first outside
   check; it is not enough on its own.

3. **Mention counting is a regex.** Synonyms and negation are handled by hand-written patterns.
   They undercount paraphrase ("heart size is at the upper limits of normal") and cannot see
   hedging ("possible early consolidation"). A CheXbert-style labeller would be better.

4. **CheXpert support labels are machine-extracted**, test labels are radiologist consensus. Label
   noise in the support set is a competing explanation for CheXpert's weaker few-shot behaviour
   and is **not** separated from frequency by this design. ChestX-ray14 has clean support labels
   and shows the same pattern, which argues for frequency but does not prove it.

5. **Two checkpoints, not one.** Each dataset selects its own, which is what makes each dataset's
   result fair — but it means cross-dataset magnitudes compare different models. Within-dataset
   results are the primary evidence.

6. **No ensemble.** A single checkpoint, so the reproduction target is 0.878, not 0.889.

7. **Correlation, not causation.** We show that frequency predicts the benefit of adaptation. A
   causal test would strip one pathology's mentions out of the training text, retrain, and check
   that its few-shot benefit rises. That is one run per pathology.

In [ ]:
print("saved to", OUT)
for f in sorted(os.listdir(OUT)):
    print(f"  {f:<42}{os.path.getsize(os.path.join(OUT,f))/1e6:>8.1f} MB")